# 01 - Thu thập dữ liệu đánh giá Laptop trên FPTShop
Notebook này crawl dữ liệu đánh giá từ FPTShop theo hướng HTTP thuần (requests), chuẩn hóa schema tương thích pipeline EDA và lưu CSV UTF-8.

In [1]:
%pip install requests pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import json
import re
import time
from datetime import datetime
from html import unescape
from urllib.parse import urljoin, urlparse, parse_qsl, urlencode
import hashlib

import pandas as pd
import requests

In [ ]:
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/json;q=0.9,*/*;q=0.8",
    "Referer": "https://fptshop.com.vn/"
}

SESSION = requests.Session()
SESSION.headers.update(HEADERS)

def with_query_param(url: str, key: str, value):
    parsed = urlparse(url)
    query = dict(parse_qsl(parsed.query, keep_blank_values=True))
    query[key] = str(value)
    return parsed._replace(query=urlencode(query)).geturl()

def collect_product_urls(
    search_url: str,
    max_pages_safety: int = 300,
    max_products: int | None = None,
    hard_cap: int | None = None,
    consecutive_empty_pages: int = 3,
    delay_seconds: float = 0.8,
):
    product_urls = []
    seen = set()
    empty_pages = 0

    for page in range(1, max_pages_safety + 1):
        page_url = with_query_param(search_url, "page", page)
        response = SESSION.get(page_url, timeout=20)
        response.raise_for_status()
        html_text = response.text

        matches = re.findall(r'href="(https://fptshop\.com\.vn/may-tinh-xach-tay/[^"]+|/may-tinh-xach-tay/[^"]+)"', html_text)
        new_count = 0

        for href in matches:
            full_url = urljoin("https://fptshop.com.vn", href.split("?")[0].strip())
            if full_url in seen:
                continue
            seen.add(full_url)
            product_urls.append(full_url)
            new_count += 1

            if max_products is not None and len(product_urls) >= max_products:
                print(f"Đã đạt max_products={max_products}.")
                return product_urls
            if hard_cap is not None and len(product_urls) >= hard_cap:
                print(f"Đã đạt hard_cap={hard_cap} để dừng an toàn.")
                return product_urls

        print(f"Trang {page}: +{new_count} URL sản phẩm, tổng {len(product_urls)}")

        if new_count == 0:
            empty_pages += 1
        else:
            empty_pages = 0

        if empty_pages >= consecutive_empty_pages:
            print(f"Dừng do {consecutive_empty_pages} trang liên tiếp không có URL mới.")
            break

        time.sleep(delay_seconds)

    return product_urls

def extract_product_code(html_text: str):
    normalized = html_text.replace('\\"', '"').replace('\\/', '/')
    match = re.search(r'"upc":\{"code":"([^"]+)"\}', normalized)
    if match:
        return match.group(1)

    match_alt = re.search(r'"sku"\s*:\s*"([0-9]{6,})"', normalized)
    if match_alt:
        return match_alt.group(1)

    return None

def _to_int(value):
    if value is None:
        return None
    try:
        return int(float(value))
    except (TypeError, ValueError):
        return None

def _extract_urls(value):
    if value is None:
        return []
    if isinstance(value, str):
        return [value.strip()] if value.strip() else []
    urls = []
    if isinstance(value, list):
        for item in value:
            if isinstance(item, str) and item.strip():
                urls.append(item.strip())
            elif isinstance(item, dict):
                for key in ("url", "image", "src", "thumb", "thumbnail"):
                    img = item.get(key)
                    if isinstance(img, str) and img.strip():
                        urls.append(img.strip())
                        break
    elif isinstance(value, dict):
        for key in ("url", "image", "src", "thumb", "thumbnail"):
            img = value.get(key)
            if isinstance(img, str) and img.strip():
                urls.append(img.strip())
                break
    return urls

def extract_product_metadata(html_text: str, item_code: str | None = None):
    metadata = {
        "item_id": item_code,
        "product_name": None,
        "brand": None,
        "price": None,
        "final_price": None,
        "rating_count_total": None,
        "image_product": None,
    }

    ldjson_blocks = re.findall(
        r'<script[^>]*type="application/ld\\+json"[^>]*>(.*?)</script>',
        html_text,
        flags=re.DOTALL | re.IGNORECASE,
    )

    for block in ldjson_blocks:
        raw = unescape(block).strip()
        if not raw:
            continue
        try:
            parsed = json.loads(raw)
        except json.JSONDecodeError:
            continue

        candidates = parsed if isinstance(parsed, list) else [parsed]
        for candidate in candidates:
            if not isinstance(candidate, dict):
                continue
            candidate_type = candidate.get("@type")
            type_list = candidate_type if isinstance(candidate_type, list) else [candidate_type]
            if "Product" not in type_list:
                continue

            metadata["product_name"] = metadata["product_name"] or candidate.get("name")

            brand = candidate.get("brand")
            if isinstance(brand, dict):
                metadata["brand"] = metadata["brand"] or brand.get("name")
            elif isinstance(brand, str):
                metadata["brand"] = metadata["brand"] or brand

            offers = candidate.get("offers")
            if isinstance(offers, dict):
                metadata["price"] = metadata["price"] or _to_int(offers.get("price"))
            elif isinstance(offers, list):
                for offer in offers:
                    if isinstance(offer, dict) and offer.get("price") is not None:
                        metadata["price"] = metadata["price"] or _to_int(offer.get("price"))
                        break

            agg = candidate.get("aggregateRating")
            if isinstance(agg, dict):
                metadata["rating_count_total"] = metadata["rating_count_total"] or _to_int(agg.get("reviewCount"))

            image_urls = _extract_urls(candidate.get("image"))
            if image_urls and not metadata["image_product"]:
                metadata["image_product"] = image_urls[0]

    normalized = html_text.replace('\\"', '"').replace('\\/', '/')

    if not metadata["product_name"]:
        name_match = re.search(r'"upc":\{"code":"[^"]+","name":"([^"]+)"', normalized)
        if name_match:
            metadata["product_name"] = name_match.group(1)

    if not metadata["brand"]:
        brand_match = re.search(r'"brand":\{"code":"[^"]+","name":"([^"]+)"', normalized)
        if brand_match:
            metadata["brand"] = brand_match.group(1)

    if metadata["final_price"] is None:
        final_price_match = re.search(r'"productAdvanceInfo":\{.*?"finalPrice":(\d+)', normalized, flags=re.DOTALL)
        if final_price_match:
            metadata["final_price"] = _to_int(final_price_match.group(1))

    if metadata["price"] is None:
        price_match = re.search(r'"productAdvanceInfo":\{.*?"price":(\d+)', normalized, flags=re.DOTALL)
        if price_match:
            metadata["price"] = _to_int(price_match.group(1))

    if metadata["image_product"] is None:
        img_match = re.search(r'"primaryImage":\{"url":"([^"]+)"\}', normalized)
        if img_match:
            metadata["image_product"] = img_match.group(1)

    if metadata["final_price"] is None:
        metadata["final_price"] = metadata["price"]

    return metadata

def extract_comment_payload(html_text: str):
    normalized = html_text.replace('\\"', '"').replace('\\/', '/')

    pattern = r'"comment":(\{.*?"totalCount":\d+.*?\})\s*,\s*"policyProduct"'
    match = re.search(pattern, normalized, flags=re.DOTALL)
    if not match:
        return {}

    comment_raw = match.group(1)
    try:
        return json.loads(comment_raw)
    except json.JSONDecodeError:
        return {}

def fetch_product_reviews(product_url: str):
    response = SESSION.get(product_url, timeout=20)
    response.raise_for_status()
    html_text = response.text

    item_code = extract_product_code(html_text)
    product_meta = extract_product_metadata(html_text=html_text, item_code=item_code)
    comment_payload = extract_comment_payload(html_text)

    review_items = comment_payload.get("items") or []
    total_count = comment_payload.get("totalCount", len(review_items))

    if product_meta.get("rating_count_total") is None:
        product_meta["rating_count_total"] = _to_int(total_count)

    return product_meta, review_items, total_count

def parse_rating_row(rating_row: dict, product_meta: dict, product_url: str, row_index: int):
    created_raw = (
        rating_row.get("createdAt")
        or rating_row.get("createAt")
        or rating_row.get("createdDate")
        or rating_row.get("created_time")
        or rating_row.get("time")
    )
    created_at = pd.to_datetime(created_raw, errors="coerce")
    created_at = created_at.isoformat() if pd.notna(created_at) else None

    user_id = (
        rating_row.get("userId")
        or rating_row.get("customerId")
        or rating_row.get("memberId")
        or rating_row.get("fullName")
        or "anonymous"
    )
    cmt_id = rating_row.get("id") or rating_row.get("commentId") or rating_row.get("_id")
    if cmt_id is None:
        fallback = f"{product_url}_{row_index}_{rating_row.get('content', '')}"
        cmt_id = hashlib.md5(fallback.encode("utf-8")).hexdigest()[:16]

    rating_star = rating_row.get("score") or rating_row.get("rating") or rating_row.get("star")
    comment = (
        rating_row.get("content")
        or rating_row.get("comment")
        or rating_row.get("commentContent")
        or ""
    )
    review_title = (
        rating_row.get("title")
        or rating_row.get("subject")
        or rating_row.get("headline")
        or rating_row.get("reviewTitle")
        or None
    )

    verified_raw = (
        rating_row.get("verified_purchase")
        or rating_row.get("verifiedPurchase")
        or rating_row.get("isVerified")
        or rating_row.get("isPurchased")
        or rating_row.get("is_buyer")
    )
    verified_purchase = None
    if verified_raw is not None:
        if isinstance(verified_raw, bool):
            verified_purchase = verified_raw
        elif isinstance(verified_raw, (int, float)):
            verified_purchase = bool(verified_raw)
        elif isinstance(verified_raw, str):
            verified_purchase = verified_raw.strip().lower() in {"true", "1", "yes", "y", "co", "có"}

    product_items = (
        rating_row.get("variantName")
        or rating_row.get("modelName")
        or rating_row.get("attributeValue")
        or ""
    )

    image_review_urls = []
    for key in ("images", "imageUrls", "image_urls", "attachments", "pictures", "photos", "media"):
        if key in rating_row:
            image_review_urls = _extract_urls(rating_row.get(key))
            if image_review_urls:
                break
    image_review = " | ".join(image_review_urls) if image_review_urls else None

    return {
        "review_id": f"{user_id}_{cmt_id}",
        "shop_id": "FPTShop",
        "item_id": product_meta.get("item_id"),
        "product_name": product_meta.get("product_name"),
        "brand": product_meta.get("brand"),
        "price": product_meta.get("price"),
        "final_price": product_meta.get("final_price"),
        "rating_count_total": product_meta.get("rating_count_total"),
        "user_id": str(user_id),
        "rating_star": rating_star,
        "review_title": review_title,
        "comment": str(comment).strip(),
        "verified_purchase": verified_purchase,
        "image_product": product_meta.get("image_product"),
        "image_review": image_review,
        "created_at": created_at,
        "like_count": rating_row.get("like") or rating_row.get("helpful") or 0,
        "product_items": str(product_items).strip(),
        "source": "FPTShop"
    }

def crawl_product_reviews(product_url: str, max_reviews: int | None = None, page_size: int = 50, delay_seconds: float = 1.2):
    product_meta, page_reviews, total_count = fetch_product_reviews(product_url=product_url)

    reviews = []
    seen_review_ids = set()

    for idx, row in enumerate(page_reviews):
        if max_reviews is not None and len(reviews) >= max_reviews:
            break

        parsed = parse_rating_row(row, product_meta=product_meta, product_url=product_url, row_index=idx)
        if not parsed["comment"]:
            continue
        if parsed["review_id"] in seen_review_ids:
            continue

        seen_review_ids.add(parsed["review_id"])
        reviews.append(parsed)

    print(f"{product_url} -> lấy {len(reviews)} review hiển thị / totalCount={total_count}")
    if total_count > len(page_reviews):
        print("Lưu ý: hiện chỉ thu được danh sách review hiển thị từ HTML, chưa phân trang sâu bằng endpoint nội bộ.")

    time.sleep(delay_seconds)
    return pd.DataFrame(reviews)

In [4]:
search_url = "https://fptshop.com.vn/tim-kiem?s=laptop&sort=noi-bat&categories=may-tinh-xach-tay"
product_urls = collect_product_urls(
    search_url=search_url,
    max_pages_safety=300,
    max_products=None,
    hard_cap=None,
    consecutive_empty_pages=3,
    delay_seconds=0.8
)

display(product_urls[:10])
print(f"Tổng URL sản phẩm laptop đã thu thập: {len(product_urls)}")

if not product_urls:
    raise ValueError("Không lấy được danh sách product_urls từ trang tìm kiếm FPTShop.")

Trang 1: +25 URL sản phẩm, tổng 25
Trang 2: +23 URL sản phẩm, tổng 48
Trang 3: +24 URL sản phẩm, tổng 72
Trang 4: +24 URL sản phẩm, tổng 96
Trang 5: +24 URL sản phẩm, tổng 120
Trang 6: +24 URL sản phẩm, tổng 144
Trang 7: +24 URL sản phẩm, tổng 168
Trang 8: +24 URL sản phẩm, tổng 192
Trang 9: +24 URL sản phẩm, tổng 216
Trang 10: +24 URL sản phẩm, tổng 240
Trang 11: +24 URL sản phẩm, tổng 264
Trang 12: +24 URL sản phẩm, tổng 288
Trang 13: +24 URL sản phẩm, tổng 312
Trang 14: +24 URL sản phẩm, tổng 336
Trang 15: +24 URL sản phẩm, tổng 360
Trang 16: +24 URL sản phẩm, tổng 384
Trang 17: +24 URL sản phẩm, tổng 408
Trang 18: +24 URL sản phẩm, tổng 432
Trang 19: +24 URL sản phẩm, tổng 456
Trang 20: +24 URL sản phẩm, tổng 480
Trang 21: +24 URL sản phẩm, tổng 504
Trang 22: +24 URL sản phẩm, tổng 528
Trang 23: +24 URL sản phẩm, tổng 552
Trang 24: +24 URL sản phẩm, tổng 576
Trang 25: +24 URL sản phẩm, tổng 600
Trang 26: +24 URL sản phẩm, tổng 624
Trang 27: +24 URL sản phẩm, tổng 648
Trang 28: +24 

['https://fptshop.com.vn/may-tinh-xach-tay/lenovo-thinkpad-x1-carbon-gen-13-u7-258v-21ns010jvn',
 'https://fptshop.com.vn/may-tinh-xach-tay/colorful-rimbook-l1-i5-13420h-a10205500050',
 'https://fptshop.com.vn/may-tinh-xach-tay/macbook-pro-14-m5-pro-2026-15cpu-16gpu-24gb-2tb',
 'https://fptshop.com.vn/may-tinh-xach-tay/msi-stealth-16-ai-b3wg-008vn-u9-386h',
 'https://fptshop.com.vn/may-tinh-xach-tay/asus-zenbook-14-ux3405ca-st629w-ultra-7-255h',
 'https://fptshop.com.vn/may-tinh-xach-tay/asus-vivobook-s16-m3607ha-sh186w-ryzen-7-260',
 'https://fptshop.com.vn/may-tinh-xach-tay/lenovo-gaming-loq-15irx10-i7-13645hx-83je01agvn',
 'https://fptshop.com.vn/may-tinh-xach-tay/macbook-air-13-m5-2026-10cpu-8gpu-16gb-512gb',
 'https://fptshop.com.vn/may-tinh-xach-tay/macbook-pro-14-m5-2026-10cpu-10gpu-32gb-2tb-nano-96w',
 'https://fptshop.com.vn/may-tinh-xach-tay/asus-vivobook-s14-m3407ga-sf030w-ryzen-ai-7-445']

Tổng URL sản phẩm laptop đã thu thập: 1000


In [5]:
all_frames = []
for url in product_urls:
    df_item = crawl_product_reviews(url, max_reviews=None, page_size=50, delay_seconds=1.2)
    if not df_item.empty:
        all_frames.append(df_item)

if not all_frames:
    raise RuntimeError("Không thu được dữ liệu. Kiểm tra lại URL hoặc thử chạy lại với mạng khác.")

df_raw = pd.concat(all_frames, ignore_index=True)
df_raw = df_raw.drop_duplicates(subset=["review_id"]).reset_index(drop=True)

display(df_raw.head())
display(df_raw.tail())
print(f"Tổng số review thu được: {len(df_raw)}")

https://fptshop.com.vn/may-tinh-xach-tay/lenovo-thinkpad-x1-carbon-gen-13-u7-258v-21ns010jvn -> lấy 0 review hiển thị / totalCount=0
https://fptshop.com.vn/may-tinh-xach-tay/colorful-rimbook-l1-i5-13420h-a10205500050 -> lấy 1 review hiển thị / totalCount=1
https://fptshop.com.vn/may-tinh-xach-tay/macbook-pro-14-m5-pro-2026-15cpu-16gpu-24gb-2tb -> lấy 0 review hiển thị / totalCount=0
https://fptshop.com.vn/may-tinh-xach-tay/msi-stealth-16-ai-b3wg-008vn-u9-386h -> lấy 0 review hiển thị / totalCount=0
https://fptshop.com.vn/may-tinh-xach-tay/asus-zenbook-14-ux3405ca-st629w-ultra-7-255h -> lấy 0 review hiển thị / totalCount=0
https://fptshop.com.vn/may-tinh-xach-tay/asus-vivobook-s16-m3607ha-sh186w-ryzen-7-260 -> lấy 0 review hiển thị / totalCount=0
https://fptshop.com.vn/may-tinh-xach-tay/lenovo-gaming-loq-15irx10-i7-13645hx-83je01agvn -> lấy 0 review hiển thị / totalCount=0
https://fptshop.com.vn/may-tinh-xach-tay/macbook-air-13-m5-2026-10cpu-8gpu-16gb-512gb -> lấy 0 review hiển thị / to

,review_id,shop_id,item_id,user_id,rating_star,comment,created_at,like_count,product_items,source
0,Anh KhoiPD DX_201234,FPTShop,00925801,Anh KhoiPD DX,4,Máy dùng tốt,None,0,,FPTShop
1,chi mai_199345,FPTShop,00925128,chi mai,5,siêu dth,None,0,,FPTShop
2,Trần Minh Huy_185078,FPTShop,00922302,Trần Minh Huy,5,"Con máy này chạy mượt mà cực kỳ, chơi game hay...",None,0,,FPTShop
3,Phan Văn Hải_184943,FPTShop,00922302,Phan Văn Hải,4,"E này thiết kế đẹp mắt lắm mọi người ạ, màu đe...",None,0,,FPTShop
4,Huỳnh Thảo Vy_183464,FPTShop,00922302,Huỳnh Thảo Vy,4,"Con máy đáng đồng tiền bát gạo, mọi người cứ y...",None,0,,FPTShop


,review_id,shop_id,item_id,user_id,rating_star,comment,created_at,like_count,product_items,source
847,Thanh Tú_46774,FPTShop,00888669,Thanh Tú,4,Máy chạy ổn nhưng quạt tản nhiệt hơi ồn khi xử...,None,0,,FPTShop
848,Mai Phương_46757,FPTShop,00888669,Mai Phương,5,"Sản phẩm chính hãng, sử dụng ổn định, tốc độ n...",None,0,,FPTShop
849,bảo ngọc_46745,FPTShop,00888669,bảo ngọc,5,"Cấu hình mạnh, xử lý nhanh, màn hình hiển thị đẹp",None,0,,FPTShop
850,Loan Trần_46719,FPTShop,00888669,Loan Trần,5,"Máy chạy mượt, thiết kế gọn nhẹ, rất phù hợp c...",None,0,,FPTShop
851,Nhi_1904,FPTShop,00888670,Nhi,None,Dạ mình đặt hàng lấy sau thì lúc lấy còn áp dụ...,None,0,,FPTShop


Tổng số review thu được: 852


In [6]:
output_path = "data/fptshop_laptop_raw.csv"
df_raw.to_csv(output_path, index=False, encoding="utf-8-sig")
print(f"Đã lưu dữ liệu thô vào: {output_path}")

Đã lưu dữ liệu thô vào: data/fptshop_laptop_raw.csv


## Ghi chú
Notebook đã bỏ giới hạn max thủ công cho số URL sản phẩm và số review mỗi sản phẩm. Dữ liệu thô được mở rộng thêm metadata để phân tích: product_name, brand, price/final_price, rating_count_total, review_title, verified_purchase, image_product, image_review (nếu có). Một số cột có thể null vì phụ thuộc dữ liệu FPTShop tại thời điểm crawl.